In [1]:
import joblib
import pandas as pd
import numpy as np


data = pd.read_csv("/kaggle/input/notebooks/mezbaussalaheen/landsat/landsat8_10m_all_bands.csv")
data.head()

,Longitude,Latitude,B3,B2,B1,B5,B4,B7,B6,QA_PIXEL
0,247610.0,2740510.0,0.059765,0.026875,0.01912,0.237883,0.037903,0.050607,0.108578,21824
1,247620.0,2740510.0,0.059765,0.026875,0.01912,0.237883,0.037903,0.050607,0.108578,21824
2,247630.0,2740510.0,0.059765,0.026875,0.01912,0.237883,0.037903,0.050607,0.108578,21824
3,247640.0,2740510.0,0.058032,0.027590,0.01945,0.214095,0.038727,0.041808,0.095735,21824
4,247650.0,2740510.0,0.058032,0.027590,0.01945,0.214095,0.038727,0.041808,0.095735,21824


In [2]:
cat = joblib.load("/kaggle/input/notebooks/mezbaussalaheen/fork-of-capstone-model/saved_models/cart.joblib")
catboost = joblib.load("/kaggle/input/notebooks/mezbaussalaheen/fork-of-capstone-model/saved_models/catboost.joblib")
xgboost = joblib.load("/kaggle/input/notebooks/mezbaussalaheen/fork-of-capstone-model/saved_models/xgboost.joblib")
lightgbm = joblib.load("/kaggle/input/notebooks/mezbaussalaheen/fork-of-capstone-model/saved_models/lightgbm.joblib")
label_encoder = joblib.load("/kaggle/input/notebooks/mezbaussalaheen/fork-of-capstone-model/saved_models/label_encoder.joblib")
scaler = joblib.load("/kaggle/input/notebooks/mezbaussalaheen/fork-of-capstone-model/saved_models/scaler.joblib")


#Feature scaling
print(scaler.feature_names_in_)

['B04' 'B03' 'B08' 'B02' 'B01' 'B12' 'B11' 'EVI' 'NDBI' 'MNDWI' 'BSI'
 'NDVI']


In [3]:
data = data.rename(columns={
    "B1":"B01",   # Coastal
    "B2":"B02",   # Blue
    "B3":"B03",   # Green
    "B4":"B04",   # Red
    "B5":"B08",   # NIR
    "B6":"B11",   # SWIR1
    "B7":"B12"    # SWIR2
})
print(data.columns)

Index(['Longitude', 'Latitude', 'B03', 'B02', 'B01', 'B08', 'B04', 'B12',
       'B11', 'QA_PIXEL'],
      dtype='object')


In [4]:
# NDVI
data["NDVI"] = (data["B08"] - data["B04"]) / (data["B08"] + data["B04"] + 1e-10)

# EVI
data["EVI"] = 2.5 * (data["B08"] - data["B04"]) / (data["B08"] + 6*data["B04"] - 7.5*data["B02"] + 1)

# NDBI
data["NDBI"] = (data["B11"] - data["B08"]) / (data["B11"] + data["B08"] + 1e-10)

# MNDWI
data["MNDWI"] = (data["B03"] - data["B11"]) / (data["B03"] + data["B11"] + 1e-10)

# BSI
data["BSI"] = ((data["B11"] + data["B04"]) - (data["B08"] + data["B02"])) / \
              ((data["B11"] + data["B04"]) + (data["B08"] + data["B02"]) + 1e-10)

In [5]:
features = ['B04','B03','B08','B02','B01','B12','B11','EVI','NDBI','MNDWI','BSI','NDVI']

In [6]:
X = scaler.transform(data[features])

pred = catboost.predict(X)

data["prediction"] = label_encoder.inverse_transform(pred)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [7]:
data.to_parquet("landsat_prediction.parquet", index=False)
print("OK")

OK


In [8]:
import matplotlib.pyplot as plt

In [9]:


labels = data["prediction"].values

In [10]:
data = data.sort_values(["Latitude","Longitude"])

In [11]:
grid = data.pivot(index="Latitude", columns="Longitude", values="prediction")
print("OK")

OK


In [12]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

In [13]:
# image show
class_labels = ["Water", "Vegetation", "Barren"] 
num_classes = len(class_labels)
cmap = plt.get_cmap("tab20", num_classes)  # discrete colormap


# plot
plt.figure(figsize=(10,8))
plt.imshow(grd, cmap=cmap)
plt.title("Land Cover Classification")
plt.axis("off")

legend_elements = [Patch(facecolor=cmap(i), label=class_labels[i]) for i in range(num_classes)]
plt.legend(handles=legend_elements, loc="upper right")
plt.show()

plt.imsave("classified_landcover.png", grid, cmap=cmap)

NameError: name 'grd' is not defined

<Figure size 1000x800 with 0 Axes>

In [ ]:
red = data.pivot(index="Latitude", columns="Longitude", values="B04")
green = data.pivot(index="Latitude", columns="Longitude", values="B03")
blue = data.pivot(index="Latitude", columns="Longitude", values="B02")

rgb = np.dstack((red, green, blue))
rgb = rgb / np.nanmax(rgb)

# prediction map
pred = data.pivot(index="Latitude", columns="Longitude", values="prediction")

colors = ["blue", "green", "yellow", "red"]  # class-wise colors
class_labels = ["Water", "Vegetation", "Barren", "Urban"]  # class names
cmap = ListedColormap(colors)

# -------------------------
# plot RGB + overlay prediction
# -------------------------
plt.figure(figsize=(10,8))
plt.imshow(rgb)
plt.imshow(pred, cmap=cmap, alpha=0.4)  # overlay with transparency
plt.axis("off")
plt.title("RGB Satellite Image + Classification")

# Legend
legend_elements = [Patch(facecolor=colors[i], label=class_labels[i]) for i in range(len(colors))]
plt.legend(handles=legend_elements, loc="upper right")
plt.show()

plt.imsave("landsat_rgb.png", rgb)
plt.imsave("classified_map.png", pred, cmap=cmap)

In [ ]:
from matplotlib import cm

In [ ]:
classes = np.unique(pred)
num_classes = len(classes)
cmap = plt.get_cmap("tab20", num_classes)
class_labels = ["Water", "Vegetation", "Barren", "Urban"]
plt.figure(figsize=(10,8))
plt.imshow(rgb)
plt.imshow(pred, cmap=cmap, alpha=0.4)
plt.axis("off")
plt.title("RGB + ML Classification")
legend_elements = [Patch(facecolor=cmap(i), label=class_labels[i]) for i in range(num_classes)]
plt.legend(handles=legend_elements, loc="upper right")
plt.show()
plt.imsave("rgb_ml_classification.png", pred, cmap=cmap)